In [1]:
from google.colab import files
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("/content/drive/MyDrive/Natural Language Processing/chatbot/user_support_chatbot_questions_answers.csv", encoding='cp1252')
df.to_json(
    "/content/drive/MyDrive/Natural Language Processing/data.json",
    orient="records",
    indent=4
)

print("JSON file saved successfully to Google Drive!")

JSON file saved successfully to Google Drive!


In [3]:

import time

with open("/content/drive/MyDrive/Natural Language Processing/chatbot/data.json", "r", encoding="utf-8") as file:
    qa_data = json.load(file)

questions = [item["Question"] for item in qa_data if item["Question"] is not None]


vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(questions)


max_rounds = 3
round_count = 0

while round_count < max_rounds:
    user_question = input("\nEnter your question: ").strip()

    if len(user_question.split()) < 2:
        print("Your question is too short. Please be more specific.")
        round_count += 1
        continue

    # -------- START TIMING (ONLY LOGIC) --------
    start_time = time.time()

    user_vector = vectorizer.transform([user_question])
    similarity_scores = cosine_similarity(user_vector, question_vectors)[0]
    print(similarity_scores)
    temp_scores = similarity_scores.tolist()
    top_indices = []
    top_scores = []

    for _ in range(3):
        max_score = max(temp_scores)
        max_index = temp_scores.index(max_score)
        top_scores.append(max_score)
        top_indices.append(max_index)
        temp_scores[max_index] = -1

    best_score = top_scores[0]

    # -------- END TIMING --------
    end_time = time.time()
    print("Processing Time:", end_time - start_time, "seconds")

    if best_score >= 0.4 and (top_scores[0] - top_scores[1]) >= 0.05:
        print("\nAnswer:", qa_data[top_indices[0]]["Answer"])
        break

    elif best_score >= 0.2:
        print("\nI am not sure what you mean. Did you want to ask:")
        for i in range(3):
            print(f"{i+1}. {qa_data[top_indices[i]]['Question']}")

        choice = input("\nEnter 1, 2, or 3 (or press Enter to rephrase): ").strip()

        if choice in ["1", "2", "3"]:
            print("\nAnswer:", qa_data[top_indices[int(choice) - 1]]["Answer"])
            break
        else:
            round_count += 1
    else:
        print("\nI could not understand your question.")
        round_count += 1
    end_time = time.time()
if round_count == max_rounds:
    print("\nNo answer can be given after multiple clarification attempts.")



Enter your question: what do you need to do for account creation
[0.47503894 0.39895018 0.         0.         0.45193882 0.
 0.         0.29821622 0.26478703 0.16320975]
Processing Time: 0.0066721439361572266 seconds

I am not sure what you mean. Did you want to ask:
1. How do I create an account?
2. How do I update my account information?
3. I forgot my password. What should I do?

Enter 1, 2, or 3 (or press Enter to rephrase): what do u need to create an account

Enter your question: 3
Your question is too short. Please be more specific.

Enter your question: what to do
[0.20202809 0.49452521 0.         0.         0.1922039  0.
 0.         0.36965879 0.22800097 0.28107103]
Processing Time: 0.0019221305847167969 seconds

Answer: Click Forgot Password on the login page and follow the instructions to reset your password.


📝  Observation Note

This chatbot implements a semantic question–answer retrieval system using TF-IDF vectorization and cosine similarity. All stored questions are first converted into numerical vectors using TF-IDF, which assigns higher importance to informative terms while reducing the impact of commonly occurring words. The vectorizer is fitted once to improve efficiency.

For each user query, the input is transformed using the same TF-IDF model and compared against all stored question vectors using cosine similarity. Since cosine similarity measures the angular distance between vectors, it effectively captures textual similarity even when exact word matches are not present.

The system applies a threshold-based decision strategy. When the similarity score is high and unambiguous, the chatbot directly returns the corresponding answer. If the similarity score indicates partial relevance or ambiguity, the chatbot provides the top three most similar questions and asks the user to clarify their intent. Very short or vague inputs are rejected to avoid unreliable responses. To prevent infinite interaction, the chatbot limits clarification attempts to three rounds, after which it terminates gracefully if no confident match is found.

In this implementation, the chatbot uses the Bag of Words (BoW) model with Cosine Similarity to retrieve the most relevant answer from a predefined question–answer dataset. Each question is converted into a numerical vector based on word frequency, and user input is matched against these vectors.

The observations show that BoW performs effectively for short, direct, and keyword-based questions, where exact or near-exact word overlap exists between the user query and stored questions. The cosine similarity scores correctly identify the most relevant question when the vocabulary closely matches.

However, BoW does not consider word importance or semantic meaning, leading to reduced accuracy when:

Synonyms are used

Word order varies

Common words dominate the sentence

As a result, multiple questions may receive similar similarity scores, requiring clarification-based interaction to resolve ambiguity. This behavior is handled using top-3 similarity ranking and threshold-based decision logic.

The processing time remains consistently low, indicating that the Bag of Words approach is computationally efficient and suitable for small to medium-sized datasets.

Overall, the Bag of Words model provides a simple, fast, and interpretable baseline solution for question matching, but its limitations highlight the need for advanced techniques such as TF-IDF or semantic embeddings for improved contextual understanding in larger or more diverse datasets.